# Optimización de DataLoaders para el Rendimiento

El entrenamiento lento de los modelos es un obstáculo frecuente en los proyectos de aprendizaje automático. Aunque es fácil suponer que el hardware es el factor limitante, los problemas de rendimiento a menudo surgen de un cuello de botella de datos, donde la potente GPU se queda esperando a que lleguen los datos para procesar. Esta ineficiencia puede prolongar significativamente los ciclos de entrenamiento. Abordar cómo se cargan y entregan los datos al acelerador es un paso vital hacia un desarrollo de modelos más rápido y efectivo.

Este laboratorio sirve como una guía práctica para superar estos desafíos mediante el uso eficaz del `DataLoader` de PyTorch. Explorarás cómo se pueden configurar sus diversos parámetros para construir un pipeline de datos eficiente que mantenga tu hardware totalmente ocupado. Al final de esta sesión, tendrás una comprensión práctica de cómo diagnosticar y resolver problemas comunes de carga de datos.

En este notebook, tú podrás:

* Investigar cómo habilitar el procesamiento de datos en paralelo puede reducir drásticamente el tiempo de inactividad de la GPU.

* Medir el impacto del **batching** en la velocidad de procesamiento y descubrir su relación con los límites de memoria del hardware.

* Experimentar con el ajuste fino de parámetros que pueden acelerar la transferencia de datos y gestionar el almacenamiento en búfer (**buffering**) para obtener ganancias de rendimiento adicionales.

## 🚨 El Objetivo: Aprender el proceso, no los números 🚨

Este notebook demuestra cómo los parámetros clave del `DataLoader` afectan el rendimiento. 
Debido a que este laboratorio opera en un **entorno restrictivo** con GPU y memoria compartida limitadas, **se adopta una metodología controlada** para aislar el impacto de cada configuración. Esto implica limpiar la memoria de forma agresiva entre ejecuciones para asegurar que cada prueba sea independiente. *(Nota: Esta limpieza es para fines de demostración y no es una práctica típica en proyectos donde el almacenamiento en caché es beneficioso).*

Los resultados y valores óptimos mostrados están hechos a medida para demostrar los conceptos **dentro de este laboratorio específico**. 
En proyectos del mundo real, la configuración ideal siempre **variará dependiendo del hardware único (CPU, GPU, RAM) y del dataset (tamaño, complejidad)**.

Por lo tanto, la conclusión clave es el **proceso de experimentación** en sí mismo. Este proceso proporciona un plano (blueprint) sobre cómo probar estos parámetros sistemáticamente y encontrar la configuración óptima.

## Imports

In [ ]:
import gc
import os
import json

import torch
from torch.utils.data import DataLoader

import helper_utils

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Medición del rendimiento

En los próximos experimentos, medirás el **tiempo total** que toma cargar el dataset completo para una época completa utilizando la función auxiliar `measure_average_epoch_time`.

Esta métrica es el indicador principal del rendimiento del `DataLoader` en este laboratorio. Al minimizar el tiempo total de la época, puedes mejorar la eficiencia general del proceso de entrenamiento.

## Ajuste de `num_workers` para la carga en paralelo

Al entrenar modelos de aprendizaje profundo, puedes enfrentarte a un **cuello de botella en la carga de datos** cuando tu potente GPU se queda inactiva esperando a que la CPU prepare los datos, lo que ralentiza todo el proceso.

Para solucionar esto, el `DataLoader` de PyTorch tiene un parámetro llamado `num_workers`. Por defecto es `0`, lo que significa que un solo proceso carga los datos; sin embargo, aumentar este valor permite que múltiples procesos preparen los datos en paralelo. 
Piensa en esto como tener a varios cocineros preparando comidas a la vez en lugar de solo uno, asegurando un flujo constante de alimentos y acelerando significativamente el pipeline.

Encontrar el número óptimo de trabajadores es un paso de optimización importante. 
Para determinar cuántos procesos paralelos puede ejecutar tu CPU para soportar las operaciones de la GPU, un buen punto de partida es verificar el número de núcleos (cores) de CPU disponibles. 

**Nota:** En última instancia, el número ideal equilibra las capacidades de tu hardware específico (CPU, GPU y velocidad del disco) con la **complejidad de tu dataset**, algo que solo puedes encontrar a través de la experimentación.

En los siguientes ejemplos, experimentarás con varios ajustes diferentes de *workers* para medir su impacto en el rendimiento utilizando el dataset CIFAR10.

* Carga el dataset *CIFAR10*.

In [ ]:
trainset = helper_utils.download_and_load_cifar10()

* Antes de comenzar a experimentar, puedes averiguar exactamente cuántos núcleos tiene tu entorno ejecutando el siguiente código.

In [ ]:
cpu_cores = os.cpu_count()
print(f"Number of available CPU cores: {cpu_cores}")

<br>

**El punto de rendimientos decrecientes**

Solo porque puedas establecer un número alto de trabajadores (a veces incluso más que los núcleos de CPU disponibles), no significa que debas hacerlo. Agregar más trabajadores está sujeto a una ley de **rendimientos decrecientes**, donde eventualmente llegas a un punto en el que agregar más aporta poco beneficio o incluso puede ralentizar las cosas. Esto suele deberse a dos factores principales:

* **Límites del sistema**: Cada trabajador requiere recursos del sistema, incluyendo **memoria compartida** (shared memory), para operar. Crear demasiados trabajadores puede agotar esta memoria, lo que podría causar que tu programa falle.


* **Competencia por recursos**: Después de cierto punto, agregar más trabajadores crea su propia sobrecarga (overhead). Puede provocar competencia por otros recursos, como la velocidad de lectura de tu disco o el propio proceso principal de Python, creando un nuevo cuello de botella.



*Tu objetivo es encontrar el "punto ideal" (sweet spot): el **número más pequeño** de trabajadores que mantenga a tu GPU constantemente suministrada con datos.*


Para comenzar tu experimento, definirás una lista para probar seis valores diferentes de `num_workers`: `0`, `2`, `4`, `6`, `8` y `10`.

In [ ]:
# Define the list of num_workers values to test
workers_to_test = [0, 2, 4, 6, 8, 10]

* Ejecuta la celda de abajo para medir el tiempo que toma cargar el dataset completo, basado en cada valor de `num_workers`, manteniendo un `batch_size=32` fijo.
* Para cada configuración, las pruebas de tiempo se ejecutan durante cinco épocas. Las primeras dos son para el **calentamiento (warm-up)**, lo que permite que los procesos del sistema se estabilicen, mientras que las **tres finales** se promedian para calcular el tiempo de carga estable.

<div style="background-color: #FFD2D2; border: 1px solid #D8000C; color: black; padding: 15px; border-radius: 5px;">
    <p>🚨 <b>NOTA IMPORTANTE:</b> Presta atención a cualquier mensaje de <code>RuntimeError</code>. Estos pueden ocurrir con un número elevado de trabajadores en sistemas con <b>memoria compartida (shared memory)</b> limitada. Si has modificado el código y encuentras este error, tu <strong>primer</strong> paso debe ser deshacer tus modificaciones, luego reiniciar el kernel y ejecutar el código original para continuar.</p>
</div>

In [ ]:
def experiment_workers(workers_to_test, trainset, device):
    """
    Mide el tiempo de carga de datos para diferentes números de trabajadores (workers).

    Args:
        workers_to_test: Una lista de enteros que representa el número de trabajadores a probar.
        trainset: El conjunto de datos que se va a cargar.
        device: El dispositivo al que se moverán los datos (por ejemplo, 'cpu' o 'cuda').
    """
    # Inicializar un diccionario para almacenar los resultados
    worker_times = {}

    # Iterar a través de cada número de trabajadores que se desea probar
    for nw in workers_to_test:
        print(f"--- Probando Número de Trabajadores = {nw} ---")
        
        # Crear una nueva instancia de DataLoader para cada prueba específica.
        loader = DataLoader(trainset, 
                            batch_size=32, 
                            shuffle=True,
                            # 'num_workers' se establece con el valor actual del bucle.
                            num_workers=nw
                        )
        
        # Manejar posibles errores de tiempo de ejecución
        try:
            # Medir el tiempo de carga de datos para una época y guardarlo en el diccionario
            worker_times[nw] = helper_utils.measure_average_epoch_time(loader, device)
        except RuntimeError as e:
            # Si ocurre un error (a menudo por quedarse sin memoria compartida)
            print(f"\n❌ ERROR con {nw} trabajadores. Probablemente un problema de memoria compartida.")
            worker_times[nw] = float('inf')
            
        # Limpiar el cargador y llamar al recolector de basura para liberar memoria
        del loader
        gc.collect()

        # Limpiar la caché de PyTorch CUDA para liberar memoria de la GPU
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return worker_times

La mayoría de los experimentos en este laboratorio se ejecutan utilizando la función auxiliar `run_experiment`. El propósito de esta función es **evitar la re-ejecución de experimentos que ya fueron exitosos** en caso de un fallo o interrupción.

Por defecto, la función primero verifica si ya existe un archivo de resultados para un experimento dado. Si no lo encuentra, lo cual sucede en la primera ejecución, realizará el cálculo y guardará el resultado. Si se encuentra un archivo de resultados en una ejecución posterior, cargará los datos de ese archivo en lugar de calcularlos nuevamente.

Los resultados de cada experimento se almacenan en la carpeta `checkpoint_experiments`, con un archivo por experimento (nombrado según el experimento) en formato JSON.


**Nota**: Si necesitas forzar que un experimento específico se ejecute de nuevo, puedes pasar el flag `rerun=True` al llamar a la función `run_experiment`. Por ejemplo: `run_experiment(..., rerun=True)`.

In [ ]:
# Ejecutar el experimento para medir el tiempo de carga de datos para diferentes números de trabajadores (workers).
worker_times = helper_utils.run_experiment(
    # Un nombre único para este experimento, usado como nombre de archivo para los resultados en caché.
    experiment_name='worker_times', 
    # La función real que contiene la lógica del experimento.
    experiment_fcn=experiment_workers, 
    # Los parámetros sobre los cuales iterar; en este caso, una lista de conteos de workers.
    cases=workers_to_test, 
    # El dataset requerido por la función del experimento.
    trainset=trainset, 
    # El dispositivo de computación (por ejemplo, 'cpu' o 'cuda') a ser utilizado.
    device=device,
    # Si es False, la función cargará los resultados desde la caché si existen.
    # Si es True, forzará al experimento a ejecutarse de nuevo y sobrescribirá cualquier resultado antiguo.
    rerun=False
)

<br>

* Con los datos de rendimiento recopilados, ahora puedes graficar los resultados para visualizar el impacto de `num_workers` en el tiempo de carga.

In [ ]:
helper_utils.plot_performance_summary(
    worker_times, 
    title="DataLoader Performance vs. num_workers", 
    xlabel="Number of Workers", 
    ylabel="Average Time per Epoch (milliseconds)"
)

<br>

**Interpretación de los Resultados**

Los tiempos exactos que veas pueden fluctuar ligeramente con cada ejecución, dependiendo de la carga actual del sistema en el entorno del notebook. Sin embargo, la tendencia general que observes debería ser clara:

* **El Salto Inicial**: Es casi seguro que verás la mejora de rendimiento más significativa al pasar de `0` a `2` trabajadores. Esto demuestra el beneficio potente e inmediato de habilitar la carga de datos en paralelo.

* **Rendimientos Decrecientes**: Más allá de 2 trabajadores, los resultados suelen volverse menos predecibles. El tiempo podría estancarse, fluctuar o incluso aumentar ligeramente con más trabajadores. Este es un ejemplo clásico de la ley de rendimientos decrecientes, donde añadir más recursos eventualmente deja de proporcionar un beneficio claro y puede crear una nueva sobrecarga (overhead).



El objetivo es encontrar una configuración confiable que sea consistentemente rápida. 
**Para este entorno**, se puede observar que `num_workers=6` suele ser una opción buena y equilibrada que balancea la velocidad con el uso de recursos.

### Visualización de la eficiencia del DataLoader

El gráfico anterior mostró el tiempo total de carga por época. Para entender mejor *por qué* algunas configuraciones son más rápidas, puedes visualizar el **desglose de eficiencia** de tu pipeline de DataLoader.

Al utilizar la función `visualize_dataloader_efficiency`, puedes obtener este desglose. En lugar de medir la época completa, realiza un **análisis por lote (per-batch analysis)** para revelar la eficiencia del pipeline.

Obtendrás un gráfico de barras normalizado. Cada barra representa el 100% del tiempo total dedicado por lote para diferentes configuraciones de `num_workers`, compuesto por:

* La parte **azul** de la barra representa el **Tiempo Activo de la GPU**: el porcentaje de tiempo dedicado al trabajo productivo (mover datos al dispositivo GPU).

* La parte **amarilla** representa el **Tiempo de Espera/Inactividad de la GPU**: el porcentaje de tiempo que la GPU pasa esperando a que la CPU prepare y entregue el siguiente lote. Este tiempo de inactividad representa el cuello de botella del pipeline que deseas minimizar.

Piensa en la GPU como un cliente en un restaurante y en los trabajadores de la CPU como chefs. La parte azul muestra qué proporción del tiempo total pasa el cliente comiendo (trabajo productivo), mientras que la parte amarilla muestra cuánto tiempo pasa esperando a que llegue el siguiente plato (tiempo perdido). *Las configuraciones más eficientes tendrán secciones azules más grandes y secciones amarillas más pequeñas.*

In [ ]:
### This cell will take a few seconds to run

# Create the dictionary of loaders iteratively using a dictionary comprehension
# for each number in the 'workers_to_test' list.
loaders_to_compare = {
    f"{nw} Workers": DataLoader(trainset, batch_size=32, num_workers=nw) 
    for nw in workers_to_test
}

# Pass the generated dictionary to the plotting function.
helper_utils.visualize_dataloader_efficiency(loaders_to_compare, device)

# Clean up and release memory.
del loaders_to_compare
gc.collect()

# Clear the PyTorch CUDA cache to free up GPU memory.
if torch.cuda.is_available():
    torch.cuda.empty_cache()

<br>

**Cómo interpretar el gráfico**

El gráfico de barras te ayuda a diagnosticar **por qué** algunas configuraciones son más rápidas al visualizar la **eficiencia** de cada configuración de `num_workers`. Muestra la proporción de tiempo que la GPU pasa realizando trabajo útil frente al tiempo que pasa esperando datos.

Debido a que el rendimiento *puede fluctuar ligeramente con cada ejecución* **en este entorno compartido**, tu objetivo es encontrar una configuración que sea consistentemente rápida y eficiente. Para este experimento, verás una mejora significativa en la eficiencia al pasar de `0` a `2` trabajadores, ya que la barra amarilla de "espera" se reduce drásticamente. Después de eso, es probable que observes rendimientos decrecientes.

Como regla general, lo que buscas es **Eficiencia de la GPU**: porcentajes azules altos y porcentajes amarillos bajos.

## Explorando el efecto de `batch_size`

Junto con el número de trabajadores, el `batch_size` (tamaño del lote) es otro parámetro fundamental para optimizar tu pipeline de datos. El procesamiento por lotes permite que tu modelo procese múltiples muestras de datos al mismo tiempo, lo cual es esencial para hacer un uso eficiente del hardware paralelo como una GPU.

Piensa en esto como un autobús en lugar de un coche; un autobús (batch size grande) transporta a muchas personas a la vez, lo que resulta en un viaje más eficiente que enviar a cada persona en su propio coche individual (batch size pequeño).

Si bien un tamaño de lote más grande a menudo puede conducir a un entrenamiento más rápido al maximizar la utilización del hardware, no está exento de límites. La restricción más significativa es la **memoria de la GPU (VRAM)**. 

A medida que aumentas el tamaño del lote, verás que los tiempos de carga mejoran y luego se estabilizan, hasta que el lote se vuelve demasiado grande y provoca un error de "fuera de memoria" (Out of Memory o OOM). En el próximo experimento, mantendrás constante el número de trabajadores y probarás diferentes tamaños de lote para observar su efecto en el rendimiento.

* Para este experimento, definirás una lista para probar seis valores diferentes de `batch_size`: `16`, `32`, `64`, `128`, `256` y `512`.

In [ ]:
# Define the list of batch_size values to test
batch_sizes_to_test = [16, 32, 64, 128, 256, 512]

* Ejecuta la celda de abajo para medir el tiempo que toma cargar el dataset completo, basado en cada valor de `batch_size`, manteniendo un valor fijo de `num_workers=6`.
* Para cada configuración, las pruebas de tiempo se ejecutan durante cinco épocas. Las primeras dos son para el **calentamiento (warm-up)**, lo que permite que los procesos del sistema se estabilicen, mientras que las **tres finales** se promedian para calcular el tiempo de carga estable.

<div style="background-color: #FFD2D2; border: 1px solid #D8000C; color: black; padding: 15px; border-radius: 5px;">
    <p>🚨 <b>NOTA IMPORTANTE:</b> Presta atención a cualquier mensaje de <code>RuntimeError</code>. Estos a menudo indican que has excedido un límite de recursos del sistema, como quedarte sin <b>memoria compartida</b> (común con un número alto de trabajadores) o sin <b>memoria de la GPU</b> (común con un tamaño de lote grande). Si has modificado el código y encuentras este error, tu <strong>primer</strong> paso debe ser deshacer tus modificaciones, luego reiniciar el kernel y ejecutar el código original para continuar.</p>
</div>

In [ ]:
def experiment_batch_sizes(batch_sizes_to_test, trainset, device):
    """
    Mide el tiempo de carga de datos para diferentes tamaños de lote (batch sizes).

    Args:
        batch_sizes_to_test: Una lista de enteros que representa los tamaños de lote a probar.
        trainset: El conjunto de datos que se va a cargar.
        device: El dispositivo al que se moverán los datos (por ejemplo, 'cpu' or 'cuda').
    """
    # Inicializar un diccionario para almacenar los resultados
    batch_size_times = {}

    # Iterar a través de cada tamaño de lote que se desea probar
    for bs in batch_sizes_to_test:
        print(f"--- Probando Tamaño de Lote = {bs} ---")
        
        # Crear una nueva instancia de DataLoader para cada prueba específica.
        loader = DataLoader(trainset, 
                            # El 'batch_size' se establece con el valor actual del bucle.
                            batch_size=bs, 
                            shuffle=True,
                            num_workers=6
                        )
        
        # Manejar posibles errores de tiempo de ejecución, especialmente los de falta de memoria (OOM).
        try:
            # Medir el tiempo de carga de datos para una época y guardarlo en el diccionario.
            batch_size_times[bs] = helper_utils.measure_average_epoch_time(loader, device)
        except RuntimeError as e:
            # Si ocurre un error (a menudo por quedarse sin memoria en la GPU).
            print(f"\n❌ ERROR con el tamaño de lote {bs}. Probablemente un problema de memoria de la GPU.")
            batch_size_times[bs] = float('inf')
            
        # Limpiar el cargador y llamar al recolector de basura para liberar memoria,
        # asegurando que cada prueba se ejecute en un entorno limpio.
        del loader
        gc.collect()

        # Limpiar la caché de PyTorch CUDA para liberar memoria de la GPU.
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
    return batch_size_times

In [ ]:
# Ejecutar el experimento para medir el tiempo de carga de datos para diferentes tamaños de lote.
batch_size_times = helper_utils.run_experiment(
    # Un nombre único para este experimento, usado como nombre de archivo para los resultados en caché.
    experiment_name="batch_size_times", 
    # La función real que contiene la lógica del experimento.
    experiment_fcn=experiment_batch_sizes,
    # Los parámetros sobre los cuales iterar; en este caso, una lista de diferentes tamaños de lote.
    cases=batch_sizes_to_test,
    # El dataset requerido por la función del experimento.
    trainset=trainset,
    # El dispositivo de computación (por ejemplo, 'cpu' o 'cuda') a ser utilizado.
    device=device,
    # Si es False, la función cargará los resultados desde la caché si existen.
    # Si es True, forzará al experimento a ejecutarse de nuevo y sobrescribirá cualquier resultado antiguo.
    rerun=False
)

<br>

* Con los datos de rendimiento recopilados, ahora puedes graficar los resultados para visualizar el impacto de `batch_size` en el tiempo de carga.

In [ ]:
helper_utils.plot_performance_summary(
    batch_size_times, 
    title="DataLoader Performance vs. batch_size", 
    xlabel="Batch Sizes", 
    ylabel="Average Time per Epoch (milliseconds)"
)

<br>

**Interpretación de los Resultados**

Nuevamente, los tiempos exactos pueden fluctuar con cada ejecución, pero la tendencia general debería ser muy clara:

* **Aceleración inicial**: Notarás una caída significativa en el tiempo de carga a medida que aumentas el `batch_size` desde valores pequeños (por ejemplo, de 16 a 128). Esto se debe a que los lotes más grandes hacen un uso mucho más eficiente de las capacidades de procesamiento paralelo de la GPU, reduciendo la sobrecarga por muestra.

* **Rendimientos decrecientes**: También puedes ver la ley de los rendimientos decrecientes en acción. La ganancia de rendimiento al duplicar el tamaño del lote de 256 a 512 es mucho menor que la ganancia al duplicarlo de 16 a 32. Esto sucede porque la GPU ya se está saturando de trabajo.

El objetivo es encontrar el tamaño de lote más grande que brinde un buen rendimiento sin exceder los límites de tu sistema. Aunque "más grande" suele ser "más rápido", existe un límite estricto. En este entorno de notebook, por ejemplo, aumentar el `batch_size` a 1024 provocará un error de memoria (Out of Memory) en la GPU.

## Optimización con `pin_memory`

Ahora que has optimizado `num_workers` y `batch_size`, puedes explorar otra optimización para acelerar la transferencia de datos de la CPU a la GPU. Si bien puede proporcionar un aumento significativo de velocidad, esta función conlleva un compromiso: **consume más memoria RAM de tu sistema**.

Por defecto, los datos cargados por la CPU se encuentran en una memoria "paginable" (pageable), lo que requiere un paso de copia adicional antes de que la GPU pueda acceder a ellos. Al establecer `pin_memory=True` en el `DataLoader`, le indicas que utilice una región de memoria especial "fijada" o "anclada" (pinned). Esto permite una transferencia de memoria más rápida y directa a la GPU.

Esto es similar a la seguridad de un aeropuerto; la memoria estándar requiere que te detengas y desempaques tus maletas en bandejas separadas, mientras que la memoria fija es como tener tus maletas preparadas para el carril exprés, lo que permite un paso mucho más rápido por el escáner.

En este experimento, tomarás tu configuración con mejor rendimiento de los pasos anteriores y verás cuánta velocidad adicional puedes ganar habilitando `pin_memory`.

* Define una lista con valores booleanos para `pin_memory`.

In [ ]:
pin_memory_settings = [False, True]

* Ejecuta la celda de abajo para medir el tiempo que toma cargar el dataset completo para cada configuración de `pin_memory` (`False` y `True`), manteniendo fijos los valores de `num_workers=6` y `batch_size=256`.

* Para cada configuración, las pruebas de tiempo se ejecutan durante cinco épocas. Las primeras dos son para el calentamiento (**warm-up**), lo que permite que los procesos del sistema se estabilicen, mientras que las **tres finales** se promedian para calcular el tiempo de carga estable.

In [ ]:
def experiment_pin_memory(pin_memory_settings, trainset, device):
    """
    Mide el tiempo de carga de datos con y sin memoria fijada (pinned memory).

    Args:
        pin_memory_settings: Una lista de valores booleanos para probar pin_memory.
        trainset: El conjunto de datos que se va a cargar.
        device: El dispositivo al que se moverán los datos (por ejemplo, 'cpu' o 'cuda').
    """
    # Inicializar un diccionario para almacenar los resultados
    pin_memory_times = {}

    # Iterar a través de cada configuración de pin_memory
    for setting in pin_memory_settings:
        print(f"--- Probando con pin_memory = {setting} ---")
        
        # Crear un DataLoader con la configuración actual de pin_memory
        loader = DataLoader(trainset,
                            batch_size=256,
                            num_workers=6,
                            shuffle=True,
                            # 'pin_memory' se establece con el valor booleano actual del bucle.
                            pin_memory=setting
                        )
        
        try:
            # Medir el rendimiento y almacenar el resultado en el diccionario
            pin_memory_times[setting] = helper_utils.measure_average_epoch_time(loader, device)
        except RuntimeError as e:
            # Imprimir un mensaje de error si ocurre una excepción
            print(f"\n❌ Ocurrió un error con pin_memory = {setting}: {e}")
            pin_memory_times[setting] = float('inf')
            
        # --- Limpieza de memoria para cada iteración ---
        del loader
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return pin_memory_times

In [ ]:
# Ejecutar el experimento para medir el tiempo de carga de datos con pin_memory establecido en False o True.
pin_memory_times = helper_utils.run_experiment(
    # Un nombre único para este experimento, usado como nombre de archivo para los resultados en caché.
    experiment_name="pin_memory_times",
    # La función real que contiene la lógica del experimento.
    experiment_fcn=experiment_pin_memory,
    # Los parámetros sobre los cuales iterar; en este caso, una lista de valores booleanos para pin memory.
    cases=pin_memory_settings,
    # El dataset requerido por la función del experimento.
    trainset=trainset, 
    # El dispositivo de computación (por ejemplo, 'cpu' o 'cuda') a ser utilizado.
    device=device,
    # Si es False, la función cargará los resultados desde la caché si existen.
    # Si es True, forzará al experimento a ejecutarse de nuevo y sobrescribirá cualquier resultado antiguo.
    rerun=False
)

In [ ]:
helper_utils.plot_performance_summary(
    pin_memory_times, 
    title="DataLoader Performance vs. pin_memory", 
    xlabel="Pin Memory", 
    ylabel="Average Time per Epoch (milliseconds)"
)

<br>

**Interpreción de los Resultados**

Aunque a menudo se recomienda `pin_memory` como una mejora de rendimiento directa, los resultados en este entorno específico muestran lo contrario. Esto resalta que **las optimizaciones no siempre son universales y deben ser probadas**.

* **El resultado contraintuitivo**: Como se ve en el gráfico, habilitar `pin_memory=True` (representado por 1 en el eje x) en realidad resultó en un tiempo por época ligeramente más lento en comparación con dejarlo desactivado (0).  
**Nota**: En el caso poco común de que tu ejecución específica muestre una ligera mejora con `pin_memory=True`, probablemente notarás que la diferencia es marginal.


* **El costo de la sobrecarga (Overhead)**: Esto demuestra un concepto importante en el ajuste de rendimiento: cada optimización tiene una sobrecarga potencial.  
Aunque "fijar" la memoria puede acelerar el paso final de transferencia de datos a la GPU, el proceso de asignar esa memoria especial en sí mismo tiene un pequeño costo. En este escenario altamente optimizado, donde "6 workers" y un "batch size grande" ya mantienen a la GPU bien alimentada, el costo de sobrecarga de fijar la memoria superó su beneficio en la velocidad de transferencia.


**La lección sobre la experimentación**: Este es un buen ejemplo de que lograr el mejor rendimiento no significa habilitar todas las funciones disponibles. La optimización es un proceso experimental para *encontrar la combinación correcta de ajustes que mejor funcione para tu hardware, software y dataset específicos*.

## Ajuste fino con `prefetch_factor`

Ya has optimizado el número de trabajadores en paralelo y has encontrado el tamaño de lote ideal y la configuración de `pin_memory`. La última perilla que puedes girar en el `DataLoader` es el `prefetch_factor`. Este parámetro controla cuántos lotes se precargan en la memoria *por cada trabajador*.

Por defecto, `prefetch_factor=2`. Esto significa que cada trabajador siempre intentará tener dos lotes listos y esperando en segundo plano. Para la mayoría de los casos de uso, esto es suficiente para ocultar la latencia de carga de datos y mantener la GPU alimentada.

El compromiso es directo: aumentar el `prefetch_factor` a veces puede suavizar los parones en la carga de datos a costa de usar más **memoria RAM del sistema**, ya que se mantendrán más lotes precargados en memoria. Piensa en esto como un búfer más grande; puede ayudar si las velocidades de carga de datos son inconsistentes, pero consume más recursos.

En este experimento final, probarás diferentes valores de `prefetch_factor` para ver si alejarse del valor predeterminado proporciona alguna ganancia de rendimiento final para tu pipeline ya optimizado.

* Para este experimento, definirás una lista para probar seis valores diferentes de `prefetch_factor`: `2`, `4`, `6`, `8`, `10` y `12`.

In [ ]:
# Define the list of prefetch_factor values to test
prefetch_factors_to_test = [2, 4, 6, 8, 10, 12]

* Ejecuta la celda de abajo para medir el tiempo que toma cargar el dataset completo para cada valor de `prefetch_factor`, utilizando la configuración óptima determinada en los experimentos anteriores (`num_workers=6`, `batch_size=256` y `pin_memory=False`).
* Para cada configuración, las pruebas de tiempo se ejecutan durante cinco épocas. Las primeras dos son para el calentamiento (**warm-up**), lo que permite que los procesos del sistema se estabilicen, mientras que las **tres finales** se promedian para calcular el tiempo de carga estable.

In [ ]:
def experiment_prefetch_factor(prefetch_factors_to_test, trainset, device):
    """
    Mide el tiempo de carga de datos para diferentes configuraciones de prefetch_factor.

    Args:
        prefetch_factors_to_test: Una lista de enteros que representa los factores de prebúsqueda a probar.
        trainset: El conjunto de datos que se va a cargar.
        device: El dispositivo al que se moverán los datos (por ejemplo, 'cpu' o 'cuda').
    """
    # Inicializar un diccionario para almacenar los resultados
    prefetch_factor_times = {}

    # Iterar a través de cada factor de prebúsqueda que se desea probar
    for pf in prefetch_factors_to_test:
        print(f"--- Probando prefetch_factor = {pf} ---")
        
        # Crear una nueva instancia de DataLoader para cada prueba específica, usando la configuración óptima
        loader = DataLoader(trainset, 
                            batch_size=256, 
                            shuffle=True,
                            num_workers=6,
                            pin_memory=False,
                            # El 'prefetch_factor' se establece con el valor actual del bucle.
                            prefetch_factor=pf
                        )
        
        # Manejar posibles errores de tiempo de ejecución
        try:
            # Medir el tiempo de carga de datos para una época y guardarlo en el diccionario
            prefetch_factor_times[pf] = helper_utils.measure_average_epoch_time(loader, device)
        except RuntimeError as e:
            # Si ocurre un error, registrarlo.
            print(f"\n❌ ERROR con prefetch_factor {pf}: {e}")
            prefetch_factor_times[pf] = float('inf')
            
        # Limpiar el cargador y llamar al recolector de basura para liberar memoria,
        # asegurando que cada prueba se ejecute en un entorno limpio.
        del loader
        gc.collect()

        # Limpiar la caché de PyTorch CUDA para liberar memoria de la GPU.
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return prefetch_factor_times

In [ ]:
# Ejecutar el experimento para medir el tiempo de carga de datos para diferentes factores de prebúsqueda (prefetch factor).
prefetch_factor_times = helper_utils.run_experiment(
    # Un nombre único para este experimento, usado como nombre de archivo para los resultados en caché.
    experiment_name="prefetch_factor_times", 
    # La función real que contiene la lógica del experimento.
    experiment_fcn=experiment_prefetch_factor,
    # Los parámetros sobre los cuales iterar; en este caso, una lista de diferentes factores de prebúsqueda.
    cases=prefetch_factors_to_test,
    # El dataset requerido por la función del experimento.
    trainset=trainset, 
    # El dispositivo de computación (por ejemplo, 'cpu' o 'cuda') a ser utilizado.
    device=device,
    # Si es False, la función cargará los resultados desde la caché si existen.
    # Si es True, forzará al experimento a ejecutarse de nuevo y sobrescribirá cualquier resultado antiguo.
    rerun=False
)

In [ ]:
helper_utils.plot_performance_summary(
    prefetch_factor_times, 
    title="DataLoader Performance vs. prefetch_factor", 
    xlabel="Prefetch Factor", 
    ylabel="Average Time per Epoch (milliseconds)"
)

<br>

**Interpretación de los Resultados**

Este experimento final explora el `prefetch_factor` e ilustra perfectamente el principio de rendimientos decrecientes en el ajuste de rendimiento.

* **El punto de rendimientos decrecientes**: Después de lograr aceleraciones significativas con `num_workers` y `batch_size`, has llegado a la etapa de ajuste fino donde las mejoras se vuelven marginales o inexistentes. Al experimentar con `prefetch_factor`, no deberías esperar ver un ganador claro y repetible. El rendimiento probablemente sea "ruidoso", con fluctuaciones menores que no apuntan a una mejora definitiva sobre el valor predeterminado.

* **El compromiso entre RAM y estabilidad**: Aumentar el `prefetch_factor` crea un búfer de datos más grande, lo que consume más memoria RAM del sistema. Si bien un búfer más grande podría, teóricamente, proteger contra parones en la carga de datos, un pipeline que ya es eficiente rara vez se beneficia. Básicamente, estás gastando más memoria por una red de seguridad más grande que tus trabajadores rápidos probablemente no necesitan.

**Saber cuándo detenerse**: La lección más importante aquí es aprender a reconocer cuándo un sistema está suficientemente optimizado. En lugar de perseguir ganancias diminutas e inestables con parámetros avanzados, a menudo es más práctico y confiable quedarse con un valor predeterminado bien elegido como `prefetch_factor=2`. Este proporciona un equilibrio excelente entre rendimiento y uso de recursos sin añadir complejidad innecesaria.

## ¡Pruébalo tú mismo!

Exploraste cuatro parámetros clave del `DataLoader` para la optimización del rendimiento: `num_workers`, `batch_size`, `pin_memory` y `prefetch_factor`.

Una observación importante fue que los dos últimos parámetros, `pin_memory` y `prefetch_factor`, tuvieron un impacto insignificante en el rendimiento. Este resultado ocurrió porque el pipeline de datos, con el dataset CIFAR10, ya estaba altamente optimizado para este entorno de laboratorio específico mediante los ajustes principales de `num_workers=6` y `batch_size=256`.


Un desafío interesante para ti ahora es experimentar y ver si se puede lograr que `pin_memory` y `prefetch_factor` tengan un impacto más significativo. Por ejemplo, considera crear un escenario donde el pipeline sea intencionalmente subóptimo. ¿Qué sucede con el benchmark de `pin_memory` si `num_workers` se establece en 0 o 2? ¿Se vuelve más relevante el `prefetch_factor` en ese caso?

Esta es una oportunidad para jugar con diferentes combinaciones y desarrollar una intuición más profunda sobre cómo interactúan estos parámetros.

<div style="background-color: #FFD2D2; border: 1px solid #D8000C; color: black; padding: 15px; border-radius: 5px;">
<p>🚨 <b>Un recordatorio amigable:</b> Recuerda trabajar dentro de las restricciones de este entorno de laboratorio para evitar errores de memoria. Si ocurre un error, la solución más sencilla es reiniciar el kernel desde el menú superior, probar una combinación diferente de ajustes y luego ejecutar las celdas de abajo nuevamente.</p>
</div>

In [ ]:
import gc
import os
import torch
from torch.utils.data import DataLoader
import helper_utils

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
trainset = helper_utils.download_and_load_cifar10()

En la celda de código, implementa la lógica donde se muestra el marcador de posición `### Add your code here`.

* `parameter_name`: Para esta variable, asigna una cadena de texto (string) que describa el parámetro del `DataLoader` que se está probando (por ejemplo, `'pin_memory'` o `'batch_size'`).

* `list_of_values_to_test`: Para esta variable, crea una lista de Python que contenga los diferentes valores con los que deseas experimentar (por ejemplo, `[True, False]` o `[16, 32, 64]`).

* `loader`: Para esta variable, instancia un `DataLoader`. Configúralo con los ajustes deseados y asegúrate de que el parámetro que se está probando se establezca utilizando la variable `current_value` del bucle.


**Nota importante: 🚨** Al crear el `DataLoader`, la variable debe llamarse exactamente `loader`. Esto es necesario porque el código de limpieza de memoria que sigue depende de este nombre de variable específico. **No modifiques el código de limpieza**, ya que es esencial para asegurar que cada experimento se ejecute de forma independiente.

In [ ]:
def custom_experiment(trainset, device):
    """
    Ejecuta un experimento personalizado para medir el rendimiento del DataLoader.

    Args:
        trainset: El conjunto de datos que se utilizará para el experimento.
        device: El dispositivo (por ejemplo, 'cpu' o 'cuda') en el que se ejecutará la prueba.

    Returns:
        Una tupla que contiene:
            - Un diccionario con los resultados de rendimiento.
            - El nombre del parámetro que fue probado.
    """
    
    # Especifica el nombre del parámetro del DataLoader que se va a probar.
    # Por ejemplo: parameter_name = 'prefetch_factor'
    
    parameter_name = ### Add your code here

    # Proporciona una lista de valores para iterar a través del parámetro especificado.
    # Por ejemplo: list_of_values_to_test = [6, 8]
    
    list_of_values_to_test = ### Add your code 

    # Inicializar un diccionario vacío para almacenar los resultados de rendimiento.
    results_dictionary = {}

    # Iterar sobre cada valor en la lista de prueba.
    for current_value in list_of_values_to_test:
        print(f"--- Probando {parameter_name} = {current_value} ---")
        
        # Configurar e instanciar el DataLoader para la iteración de prueba actual.
        # Por ejemplo: loader = DataLoader(trainset, 
                                       #  batch_size=64, 
                                       #  shuffle=True, 
                                       #  num_workers=2, 
                                       #  pin_memory=False,
                                       #  prefetch_factor=current_value
                                       # )
        
        loader = ## Add your code here
        
        # Medir el rendimiento y manejar posibles errores de tiempo de ejecución.
        try:
            # Calcular el tiempo promedio por época y guardarlo en el diccionario de resultados.
            results_dictionary[current_value] = helper_utils.measure_average_epoch_time(loader, device)
        except RuntimeError as e:
            # Manejar casos donde ocurre un error de tiempo de ejecución, como un problema de falta de memoria.
            print(f"\n❌ ERROR con {parameter_name} = {current_value}: {e}")
            results_dictionary[current_value] = float('inf')
            
        # Asegurar que cada ejecución de prueba sea independiente limpiando la memoria.
        # Eliminar la instancia del DataLoader para liberar recursos.
        del loader
        # Invocar el recolector de basura para liberar memoria no referenciada.
        gc.collect()

        # Limpiar la caché de CUDA si hay una GPU disponible.
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Devolver el diccionario de resultados y el nombre del parámetro probado.
    return results_dictionary, parameter_name

In [ ]:
results_dictionary, parameter_name = custom_experiment(trainset=trainset, device=device)

helper_utils.plot_performance_summary(
    results_dictionary, 
    title=f"DataLoader Performance vs. {parameter_name}", 
    xlabel=parameter_name.replace('_', ' ').title(), 
    ylabel="Average Time per Epoch (milliseconds)"
)

## Conclusión

Este laboratorio demostró el proceso de optimizar sistemáticamente los parámetros del `DataLoader` de PyTorch para mejorar el rendimiento de la carga de datos. A través de una serie de experimentos controlados, investigaste el impacto de `num_workers`, `batch_size`, `pin_memory` y `prefetch_factor` en el tiempo total requerido para cargar el dataset CIFAR10 durante una época.

La ganancia de rendimiento más significativa se logró al aumentar `num_workers` de 0 a 2, resaltando el beneficio inmediato de la carga de datos en paralelo. Aumentos adicionales en `num_workers` y `batch_size` mostraron un patrón claro de rendimientos decrecientes, donde las mejoras de rendimiento se estabilizaron y, eventualmente, podrían incluso degradarse debido a la competencia por los recursos y la sobrecarga del sistema.

En última instancia, esta investigación confirma que no existe una única "mejor" configuración para un `DataLoader`. 
*Los ajustes óptimos dependen en gran medida de la interacción específica entre el hardware, la complejidad del conjunto de datos y los parámetros mismos*. 

La conclusión clave no son los valores específicos encontrados en este entorno, sino la metodología experimental en sí. Al aislar y probar sistemáticamente cada parámetro, uno puede identificar y eliminar eficazmente los cuellos de botella en la carga de datos, encontrando así la configuración más eficiente para cualquier escenario de entrenamiento dado.